In [ ]:
import pandas as pd
import geopandas as gpd
import duckdb

In [ ]:
#connect to duckdb
con = duckdb.connect('../database/spatial-db.db', read_only=False)

In [ ]:
#install the spatial extension
con.install_extension("spatial")
con.load_extension("spatial")

# State

In [ ]:
states = gpd.read_file('data/00_state/state.shp')

In [ ]:
states.head()

In [ ]:
states = states.copy() 
states["centroid"] = states["geometry"].centroid

In [ ]:
def find_neighbors(row, gdf):
    # "touches" checks if boundaries touch (i.e., share at least one point)
    # We filter out the state itself by comparing indices
    return gdf[gdf.geometry.touches(row.geometry)].index

# Collect neighbor indices for each state
neighbors_dict = {}
for idx, row in states.iterrows():
    neighbors_dict[idx] = find_neighbors(row, states)

# Store the neighbor indices in a column for convenience
states["neighbor_indices"] = states.index.map(neighbors_dict)

In [ ]:
def get_direction(centroid1, centroid2):
    """
    Classify the direction of centroid2 relative to centroid1
    as one of: 'north', 'south', 'west', 'east'.
    """
    dx = centroid2.x - centroid1.x
    dy = centroid2.y - centroid1.y
    
    # Compare absolute differences to decide axis of dominance
    if abs(dy) > abs(dx):
        return "north" if dy > 0 else "south"
    else:
        return "east" if dx > 0 else "west"

In [ ]:
directions = ["north", "south", "west", "east"]

results = []
for idx, row in states.iterrows():
    c1 = row["centroid"]
    neighbor_idxs = row["neighbor_indices"]
    
    # Track best neighbor in each direction as (neighbor_name, distance_so_far)
    best_in_dir = {d: (None, float("inf")) for d in directions}
    
    # Check all neighbors
    for n_idx in neighbor_idxs:
        neighbor_row = states.loc[n_idx]
        c2 = neighbor_row["centroid"]
        d = get_direction(c1, c2)
        
        # Compute actual distance (using centroid points)
        dist = c1.distance(c2)
        
        # If it's smaller than the best we have so far, update
        if dist < best_in_dir[d][1]:
            best_in_dir[d] = (neighbor_row["state_name"], dist)
    
    # Build final list of 4 directions: [north, south, west, east]
    final_list = []
    for d in directions:
        neighbor_name, _ = best_in_dir[d]
        if neighbor_name is None:
            final_list.append("none")
        else:
            final_list.append(neighbor_name)
    
    results.append(final_list)

states["neighbors_cardinal"] = results

In [ ]:
states.loc[states['state_name'] == 'Connecticut', 'neighbors_cardinal']

In [ ]:
states.at[5, 'neighbors_cardinal'] = ['Massachusetts', 'New York', 'New York', 'Rhode Island']

In [ ]:
states.loc[states['state_name'] == 'Pennsylvania', 'neighbors_cardinal']

In [ ]:
states.at[36, 'neighbors_cardinal'] = ['New York', 'Maryland', 'Ohio', 'New Jersey']

In [ ]:
states.loc[states['state_name'] == 'Virginia', 'neighbors_cardinal']

In [ ]:
states.at[44, 'neighbors_cardinal'] = ['Maryland', 'North Carolina', 'West Virginia', 'none']

In [ ]:
states.loc[states['state_name'] == 'West Virginia', 'neighbors_cardinal']

In [ ]:
states.at[46, 'neighbors_cardinal'] = ['Pennsylvania', 'Virginia', 'Ohio', 'Virginia']

In [ ]:
states.loc[states['state_name'] == 'District of Columbia', 'neighbors_']

In [ ]:
states.at[7, 'neighbors_cardinal'] = ['Maryland', 'Virginia', 'Virginia', 'Maryland']

In [ ]:
states.loc[states['state_name'] == 'South Carolina', 'neighbors_cardinal']

In [ ]:
states.at[38, 'neighbors_cardinal'] = ['North Carolina', 'Georgia', 'Georgia', 'none']

In [ ]:
states.loc[states['state_name'] == 'North Carolina', 'neighbors_cardinal']

In [ ]:
states.at[31, 'neighbors_cardinal'] = ['Virginia', 'South Carolina', 'Tennessee', 'none']

In [ ]:
states.loc[states['state_name'] == 'Maryland', 'neighbors_cardinal']

In [ ]:
states.at[18, 'neighbors_cardinal'] = ['Pennsylvania', 'Virginia', 'District of Columbia', 'none']

In [ ]:
states.loc[states['state_name'] == 'Illinois', 'neighbors_cardinal']

In [ ]:
states.at[11, 'neighbors_cardinal'] = ['Wisconsin', 'Kentucky', 'Missouri', 'Indiana']

In [ ]:
states.loc[states['state_name'] == 'Mississippi', 'neighbors_cardinal']

In [ ]:
states.at[22, 'neighbors_cardinal'] = ['Tennessee', 'none', 'Louisiana', 'Alabama']

In [ ]:
states.head()

In [ ]:
states.drop(columns=['centroid', 'neighbor_indices', 'neighbors_'], inplace=True)

In [ ]:
states = states[['GEOID','state_name', 'ppl_densit','transit_to','walk_to_wo','c_lat','c_lon','neighbors_cardinal', 'geometry']]

In [ ]:
states.head()

In [ ]:
# gdf = gpd.read_file('data/00_state/state.shp')
# #sort by GEOID
# gdf = gdf.sort_values(by='GEOID')
# # reset the index
# gdf = gdf.reset_index(drop=True)
# save to shapefile
states.to_file('data/00_state/state.shp')

In [ ]:
#read the shapefile into duckdb
con.sql("SELECT * FROM ST_Read('data/00_state/state.shp')")

In [ ]:
#drop table if it exists
con.execute("DROP TABLE IF EXISTS state")

In [ ]:
#create a table for the state shapefile
con.sql("CREATE TABLE state AS SELECT * FROM ST_Read('data/00_state/state.shp')")

In [ ]:
#check if the table was created
con.table('state')

In [ ]:
#get all the columns in the table
con.table('state').columns

In [ ]:
con.close()

# County

In [ ]:
county = gpd.read_file('data/01_county/full_county.shp')

In [ ]:
county

In [ ]:
# Make a copy to avoid mutating the original:
counties = county.copy()

# ------------------------------------------------------------------------
# 2. Compute centroid of each county polygon
# ------------------------------------------------------------------------
counties["centroid"] = counties.geometry.centroid

# ------------------------------------------------------------------------
# 3. Build the spatial index
# ------------------------------------------------------------------------
counties_sindex = counties.sindex  # uses rtree or pygeos-based index

# ------------------------------------------------------------------------
# 4. Define a function to get all neighbors in the same state
#    whose geometry "touches" the boundary of the given county
# ------------------------------------------------------------------------
def find_neighbors_in_state(row, gdf, sindex):
    """
    For a given county row, find all counties in the same state
    whose polygon 'touches' this county's polygon.
    """
    # 1) Filter by bounding box + 'touches' via the new .query(..., predicate="touches")
    #    This returns an index of candidate geometry that touches row's geometry
    possible_idxs = sindex.query(row.geometry, predicate="touches")
    
    # 2) Narrow to same state and exclude the row's own index
    possible_neighbors = gdf.loc[possible_idxs]
    neighbors = possible_neighbors[
        (possible_neighbors["state_name"] == row["state_name"]) &
        (possible_neighbors.index != row.name)
    ]
    return neighbors.index

# ------------------------------------------------------------------------
# 5. Create a dictionary mapping each county index -> list of neighbor indices
# ------------------------------------------------------------------------
neighbor_dict = {}
for idx, row in counties.iterrows():
    neighbor_dict[idx] = find_neighbors_in_state(row, counties, counties_sindex)

# Optional: store the neighbor indices in a column
counties["neighbor_indices"] = counties.index.map(neighbor_dict)

# ------------------------------------------------------------------------
# 6. Define a helper to classify direction: north, south, west, or east
# ------------------------------------------------------------------------
def get_direction(centroid1, centroid2):
    """
    Return 'north', 'south', 'west', or 'east' based on which axis
    has the larger difference. (Diagonal counties get lumped into
    whichever axis is dominant.)
    """
    dx = centroid2.x - centroid1.x
    dy = centroid2.y - centroid1.y
    if abs(dy) > abs(dx):
        return "north" if dy > 0 else "south"
    else:
        return "east" if dx > 0 else "west"

# We'll keep a fixed order of directions:
directions = ["north", "south", "west", "east"]

# ------------------------------------------------------------------------
# 7. For each county, pick the closest neighbor in each of the 4 directions
# ------------------------------------------------------------------------
all_neighbors_cardinal = []
for idx, row in counties.iterrows():
    this_centroid = row["centroid"]
    neighbor_idxs = row["neighbor_indices"]
    
    # Track best neighbor for each direction as (name, distance)
    best_in_dir = {d: (None, float("inf")) for d in directions}
    
    # Check each neighbor's centroid
    for n_idx in neighbor_idxs:
        neighbor_row = counties.loc[n_idx]
        nb_centroid = neighbor_row["centroid"]
        
        d = get_direction(this_centroid, nb_centroid)
        dist = this_centroid.distance(nb_centroid)
        
        # Update if this neighbor is closer in that direction
        if dist < best_in_dir[d][1]:
            best_in_dir[d] = (neighbor_row["county_nam"], dist)
    
    # Build a final 4-element list in order [north, south, west, east]
    final_list = []
    for d in directions:
        neighbor_name, _ = best_in_dir[d]
        final_list.append(neighbor_name if neighbor_name else "none")
    
    all_neighbors_cardinal.append(final_list)

# ------------------------------------------------------------------------
# 8. Store the 4-neighbor list in a new column
# ------------------------------------------------------------------------
counties["neighbors_cardinal"] = all_neighbors_cardinal

# Done! Now each row in 'counties' has an array like:
#   ["County A", "County B", "none", "County C"]
# representing the nearest neighbor to the north, south, west, and east
# (if any) within the same state.


In [ ]:
counties.head()

In [ ]:
counties = counties[['GEOID','ppl_densit','transit_to','walk_to_wo','state_name','county_nam','c_lat','c_lon','neighbors_cardinal', 'geometry']]

In [ ]:
#save to shapefile
counties.to_file('data/01_county/full_county.shp')

In [ ]:
#drop table if it exists
con.execute("DROP TABLE IF EXISTS county")

In [ ]:
#create a table for the county shapefile
con.sql("CREATE TABLE county AS SELECT * FROM ST_Read('data/01_county/full_county.shp')")
#check if the table was created
con.table('county')

In [ ]:
con.close()